智能代码审查助手
1.单agent还是多agent --> 单agent
2.agent的prompt
3.代码如何上传 --> 传入字符串形式
4.结果的返回形式 --> 问题处, 什么问题, 优化方案

In [3]:
from hello_agents import SimpleAgent, HelloAgentsLLM
import os
from dotenv import load_dotenv

#加载资源
load_dotenv()

llm = HelloAgentsLLM(
	model=os.getenv("LLM_MODEL_ID"),
	api_key=os.getenv("LLM_API_KEY"),
	base_url=os.getenv("LLM_BASE_URL") 
)

coderAgent = SimpleAgent(
	name="代码审查助手",
	llm=llm,
	system_prompt="你是一位代码审查助手,根据代码文件返回代码的问题以及优化方案",
)
code = """
def complex_function(a, b, c, d):
if a > b:
if c > d:
# 复杂逻辑
pass
else:
# 复杂逻辑
pass
else:
# 复杂逻辑
pass
"""
response = coderAgent.run(code)
print(response)

这是一段典型的“条件嵌套型”代码片段。虽然目前只是占位符，但已经暴露出多个在工程实践中需要警惕的问题。以下是详细审查意见与优化方案：

### 🔍 核心问题诊断
| 问题类别 | 具体表现 | 潜在风险 |
|:---|:---|:---|
| **语法错误** | 缩进完全缺失/混乱 | 直接运行会抛出 `IndentationError` |
| **圈复杂度高** | 多层 `if-else` 嵌套 | 测试路径呈 $2^n$ 增长，极易遗漏分支；阅读心智负担重 |
| **命名反模式** | 参数名为 `a, b, c, d` | 无法体现业务语义，后续维护者需反复对照上下文 |
| **缺乏契约规范** | 无类型提示、无 Docstring、无 `return` | 不符合现代 Python 工程规范，IDE 无法提供智能提示，接口契约模糊 |
| **职责不单一** | “复杂逻辑”直接写在条件块内 | 函数臃肿，难以复用，单元测试编写成本高 |

---

### 🛠️ 优化方案
1. **修复缩进**：统一使用 `4个空格` 作为缩进单位。
2. **卫语句(Guard Clauses)扁平化**：将“不满足条件”的情况提前返回/拦截，让主流程保持线性。
3. **逻辑下沉(SRP)**：将每个分支的具体实现抽离为独立函数，主函数仅负责**路由调度**。
4. **增强可读性**：替换语义化参数名，补充类型注解与文档字符串，明确返回值结构。

---

### 💡 重构代码示例
```python
from typing import Any

def process_conditions(
    val_a: float,
    val_b: float,
    val_c: float,
    val_d: float
) -> dict[str, Any]:
    """
    根据四组输入值的比较关系执行对应业务逻辑。

    Args:
        val_a: 第一层比较的左侧值
        val_b: 第一层比较的右侧值
        val_c: 第二层比较的左侧值
        val_d: 第二层比较的右侧值

    Returns:
        包含执行路径标识与处理结果的字典
    """